# 06 - Narrative Generation

Generate qualitative bout preview narratives from model predictions.

**Narrative Components:**
- Predicted winner and confidence level
- Expected bout type (pushing match, belt battle, evasive bout)
- Most likely winning techniques
- Closeness assessment
- Pressure situation context
- Head-to-head history

**Inputs:**
- `features.parquet` from notebook 03
- `winner_model.lgb` from notebook 04
- `kimarite_model.lgb` from notebook 04

**Outputs:**
- Sample narrative bout previews

In [ ]:
# Environment setup
import sys
import os

INPUT_PATH = '/kaggle/input/sumo-data-04' if os.path.exists('/kaggle/input') else './output'
FEATURES_PATH = '/kaggle/input/sumo-data-03' if os.path.exists('/kaggle/input') else './output'
RIKISHI_PATH = '/kaggle/input/sumo-data-01' if os.path.exists('/kaggle/input') else './output'
OUTPUT_PATH = '/kaggle/working' if os.path.exists('/kaggle/working') else './output'

if not os.path.exists('/kaggle/input'):
    sys.path.insert(0, '../src')

print(f"Models path: {INPUT_PATH}")
print(f"Features path: {FEATURES_PATH}")

In [ ]:
!pip install -q lightgbm

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

## Load Models and Data

In [ ]:
# Load models
winner_model = lgb.Booster(model_file=f"{INPUT_PATH}/winner_model.lgb")
kimarite_model = lgb.Booster(model_file=f"{INPUT_PATH}/kimarite_model.lgb")
kimarite_encoder = joblib.load(f"{INPUT_PATH}/kimarite_encoder.joblib")
feature_cols = pd.read_csv(f"{INPUT_PATH}/feature_columns.csv")['0'].tolist()

print("Models loaded")
print(f"Kimarite classes: {kimarite_encoder.classes_}")

In [ ]:
# Load features
features_df = pd.read_parquet(f"{FEATURES_PATH}/features.parquet")
print(f"Loaded {len(features_df):,} bouts")

# Load rikishi for names
try:
    rikishi_df = pd.read_parquet(f"{RIKISHI_PATH}/rikishi.parquet")
    name_lookup = dict(zip(rikishi_df['id'], rikishi_df['shikonaEn']))
    print(f"Loaded {len(name_lookup)} wrestler names")
except FileNotFoundError:
    name_lookup = {}
    print("Rikishi names not available")

## Kimarite Categories and Descriptions

In [ ]:
KIMARITE_CATEGORIES = {
    'push': ['oshidashi', 'tsukidashi', 'oshitaoshi', 'tsukiotoshi', 'tsukitaoshi', 'okuridashi'],
    'grapple': ['yorikiri', 'uwatenage', 'shitatenage', 'sukuinage', 'kotenage', 'yoritaoshi',
                'uwatedashinage', 'makiotoshi', 'kimedashi', 'sotogake', 'uchigake'],
    'evasion': ['hatakikomi', 'hikiotoshi', 'katasukashi']
}

KIMARITE_DESCRIPTIONS = {
    'yorikiri': 'force out',
    'oshidashi': 'push out',
    'hatakikomi': 'slap down',
    'uwatenage': 'overarm throw',
    'oshitaoshi': 'push down',
    'shitatenage': 'underarm throw',
    'tsukiotoshi': 'thrust down',
    'hikiotoshi': 'hand pull down',
    'kotenage': 'arm lock throw',
    'sukuinage': 'scoop throw',
    'tsukidashi': 'thrust out',
    'okuridashi': 'rear push out',
    'yoritaoshi': 'frontal force down',
    'katasukashi': 'under-shoulder swing down',
    'sotogake': 'outside leg trip',
    'uwatedashinage': 'pulling overarm throw',
    'makiotoshi': 'twist down',
    'tsukitaoshi': 'thrust and push down',
    'kimedashi': 'arm bar force out',
    'uchigake': 'inside leg trip',
}

## Narrative Generation Functions

In [ ]:
def get_confidence_description(p_favorite: float) -> Tuple[str, str]:
    """Convert win probability to confidence description."""
    if p_favorite >= 0.78:
        return "heavy favorite", "should beat"
    elif p_favorite >= 0.68:
        return "solid favorite", "is favored against"
    elif p_favorite >= 0.60:
        return "favored", "is likely to beat"
    elif p_favorite >= 0.55:
        return "slight favorite", "has the edge over"
    elif p_favorite >= 0.52:
        return "narrow favorite", "is narrowly favored over"
    else:
        return "coin flip", "faces"


def get_bout_type(category_probs: Dict[str, float]) -> str:
    """Describe expected bout type."""
    max_cat, max_prob = max(category_probs.items(), key=lambda x: x[1])
    
    if max_cat == 'push':
        if max_prob > 0.5:
            return "Expect a pushing/thrusting contest."
        return "Pushing attacks likely at the tachiai."
    elif max_cat == 'grapple':
        if max_prob > 0.5:
            return "Expect a belt battle."
        return "Will likely come down to who gets the better grip."
    else:
        return "Watch for evasive maneuvers — someone may sidestep or pull down."


def get_kimarite_description(kim_probs: Dict[str, float], top_n: int = 3) -> str:
    """Describe most likely techniques."""
    sorted_kim = sorted(kim_probs.items(), key=lambda x: x[1], reverse=True)
    top = [(k, p) for k, p in sorted_kim[:top_n] if p > 0.10]
    
    if not top:
        return ""
    
    desc = [f"{k} ({KIMARITE_DESCRIPTIONS.get(k, k)})" for k, p in top]
    
    if len(desc) == 1:
        return f"Most likely finish: {desc[0]}."
    else:
        return f"Likely finishes: {', '.join(desc[:-1])}, or {desc[-1]}."


def get_pressure_context(features: Dict, wrestler_name: str, prefix: str) -> Optional[str]:
    """Generate pressure situation narrative."""
    contexts = []
    
    if features.get(f'{prefix}_needs_one_win_for_kachikoshi'):
        day = features.get('day_of_tournament', 0)
        if day == 15:
            contexts.append(f"{wrestler_name} is fighting for kachikoshi on the final day (7-7 record)")
        else:
            contexts.append(f"{wrestler_name} needs one more win for kachikoshi")
    
    if features.get(f'{prefix}_already_makekoshi'):
        contexts.append(f"{wrestler_name} has already locked in a losing record")
    
    if features.get(f'{prefix}_is_yokozuna'):
        if features.get(f'{prefix}_yokozuna_losing_record_so_far'):
            contexts.append(f"{wrestler_name} is struggling as yokozuna")
    
    if features.get(f'{prefix}_is_ozeki'):
        contexts.append(f"{wrestler_name} is competing as ozeki")
    
    return ". ".join(contexts) + "." if contexts else None


def get_h2h_context(features: Dict, east_name: str, west_name: str) -> Optional[str]:
    """Generate head-to-head narrative."""
    h2h_total = features.get('east_h2h_total_bouts', 0)
    
    if features.get('east_h2h_never_met') or h2h_total == 0:
        return "This is their first meeting."
    
    h2h_wins = features.get('east_h2h_wins', 0)
    h2h_losses = features.get('east_h2h_losses', 0)
    streak = features.get('east_h2h_current_streak', 0)
    
    if h2h_total >= 5:
        if h2h_wins > h2h_losses + 3:
            return f"{east_name} dominates this matchup historically ({h2h_wins}-{h2h_losses})."
        elif h2h_losses > h2h_wins + 3:
            return f"{west_name} dominates this matchup historically ({h2h_losses}-{h2h_wins})."
    
    if abs(streak) >= 3:
        if streak > 0:
            return f"{east_name} has won {streak} straight in this matchup."
        else:
            return f"{west_name} has won {abs(streak)} straight in this matchup."
    
    if h2h_total >= 3:
        return f"Head-to-head record: {east_name} leads {h2h_wins}-{h2h_losses}." if h2h_wins > h2h_losses else \
               f"Head-to-head record: {west_name} leads {h2h_losses}-{h2h_wins}." if h2h_losses > h2h_wins else \
               f"Head-to-head record is even at {h2h_wins}-{h2h_losses}."
    
    return None


def get_rating_context(features: Dict, east_name: str, west_name: str) -> Optional[str]:
    """Generate rating vs rank narrative."""
    contexts = []
    threshold = 75
    
    east_diff = features.get('east_elo_minus_expected', 0)
    west_diff = features.get('west_elo_minus_expected', 0)
    
    if east_diff > threshold:
        contexts.append(f"{east_name} is performing above his rank")
    elif east_diff < -threshold:
        contexts.append(f"{east_name} appears overranked based on recent form")
    
    if west_diff > threshold:
        contexts.append(f"{west_name} is performing above his rank")
    elif west_diff < -threshold:
        contexts.append(f"{west_name} appears overranked based on recent form")
    
    return ". ".join(contexts) + "." if contexts else None

In [ ]:
def generate_narrative(row: pd.Series, 
                       winner_prob: float,
                       kim_probs: Dict[str, float],
                       name_lookup: Dict[int, str]) -> str:
    """Generate complete bout preview narrative."""
    
    # Get wrestler names
    east_id = row.get('eastId')
    west_id = row.get('westId')
    east_name = name_lookup.get(east_id, f"East ({east_id})")
    west_name = name_lookup.get(west_id, f"West ({west_id})")
    
    # Determine favorite
    if winner_prob >= 0.5:
        favorite, underdog = east_name, west_name
        p_favorite = winner_prob
    else:
        favorite, underdog = west_name, east_name
        p_favorite = 1 - winner_prob
    
    # Build narrative
    parts = []
    
    # 1. Main prediction
    conf_desc, verb = get_confidence_description(p_favorite)
    if p_favorite >= 0.52:
        parts.append(f"**{favorite}** {verb} {underdog} ({p_favorite*100:.0f}% confidence).")
    else:
        parts.append(f"**Coin flip** between {east_name} and {west_name}. Could go either way.")
    
    # 2. Bout type from kimarite categories
    cat_probs = {'push': 0, 'grapple': 0, 'evasion': 0}
    for kim, prob in kim_probs.items():
        for cat, members in KIMARITE_CATEGORIES.items():
            if kim in members:
                cat_probs[cat] += prob
                break
    
    parts.append(get_bout_type(cat_probs))
    
    # 3. Likely kimarite
    kim_desc = get_kimarite_description(kim_probs)
    if kim_desc:
        parts.append(kim_desc)
    
    # 4. H2H context
    h2h = get_h2h_context(row.to_dict(), east_name, west_name)
    if h2h:
        parts.append(h2h)
    
    # 5. Pressure context
    features = row.to_dict()
    east_pressure = get_pressure_context(features, east_name, 'east')
    west_pressure = get_pressure_context(features, west_name, 'west')
    if east_pressure:
        parts.append(east_pressure)
    if west_pressure:
        parts.append(west_pressure)
    
    # 6. Rating context
    rating_ctx = get_rating_context(features, east_name, west_name)
    if rating_ctx:
        parts.append(rating_ctx)
    
    return " ".join(parts)

## Generate Sample Narratives

In [ ]:
# Get recent bouts for demo
recent = features_df.sort_values('bashoId', ascending=False).head(1000)

# Filter to valid bouts
recent = recent[
    recent['east_won'].notna() & 
    (recent['east_career_total_bouts'] > 50) &
    (recent['west_career_total_bouts'] > 50)
].copy()

print(f"Sample bouts for narrative generation: {len(recent)}")

In [ ]:
# Prepare features for prediction
X_sample = recent[feature_cols].fillna(0)

# Get predictions
winner_probs = winner_model.predict(X_sample)
kimarite_probs = kimarite_model.predict(X_sample)

print(f"Predictions generated: {len(winner_probs)}")

In [ ]:
# Generate narratives for sample bouts
sample_size = 10

# Select diverse samples (different confidence levels)
indices = []

# High confidence picks
high_conf = np.where((winner_probs > 0.75) | (winner_probs < 0.25))[0]
if len(high_conf) > 0:
    indices.extend(np.random.choice(high_conf, min(3, len(high_conf)), replace=False))

# Medium confidence picks
med_conf = np.where(((winner_probs > 0.60) & (winner_probs < 0.75)) | 
                    ((winner_probs > 0.25) & (winner_probs < 0.40)))[0]
if len(med_conf) > 0:
    indices.extend(np.random.choice(med_conf, min(4, len(med_conf)), replace=False))

# Coin flip picks
coin_flip = np.where((winner_probs > 0.45) & (winner_probs < 0.55))[0]
if len(coin_flip) > 0:
    indices.extend(np.random.choice(coin_flip, min(3, len(coin_flip)), replace=False))

print(f"Selected {len(indices)} diverse bouts for narratives")

In [ ]:
# Generate and display narratives
print("=" * 80)
print("SAMPLE BOUT PREVIEW NARRATIVES")
print("=" * 80)

for i, idx in enumerate(indices):
    row = recent.iloc[idx]
    
    # Build kimarite probs dict
    kim_probs = {k: kimarite_probs[idx, j] for j, k in enumerate(kimarite_encoder.classes_)}
    
    # Generate narrative
    narrative = generate_narrative(row, winner_probs[idx], kim_probs, name_lookup)
    
    # Get actual outcome for comparison
    actual_winner = "East" if row['east_won'] == 1 else "West"
    predicted_winner = "East" if winner_probs[idx] > 0.5 else "West"
    correct = "✓" if actual_winner == predicted_winner else "✗"
    
    print(f"\n--- Bout {i+1} ({row['bashoId']} Day {row['day']}) ---")
    print(f"\n{narrative}")
    print(f"\n[Actual outcome: {actual_winner} won. Prediction: {correct}]")
    print("-" * 60)

## Example Narrative Templates

In [ ]:
# Show example narrative structures
print("""
NARRATIVE STRUCTURE EXAMPLES:

1. HIGH CONFIDENCE (>75%):
   "Terunofuji should beat Takakeisho (78% confidence). Expect a belt battle. 
   Most likely finish: yorikiri (force out). Terunofuji dominates this matchup 
   historically (8-2). Terunofuji is competing as yokozuna."

2. MEDIUM CONFIDENCE (60-75%):
   "Hoshoryu is favored against Wakatakakage (65% confidence). Will likely 
   come down to who gets the better grip. Likely finishes: uwatenage 
   (overarm throw), or yorikiri (force out). Wakatakakage has won 3 straight 
   in this matchup."

3. SLIGHT EDGE (55-60%):
   "Kotonowaka has the edge over Abi (58% confidence). Expect a pushing/
   thrusting contest. Most likely finish: oshidashi (push out). This is 
   their first meeting."

4. COIN FLIP (<55%):
   "Coin flip between Onosho and Midorifuji. Could go either way. Watch 
   for evasive maneuvers. Onosho is fighting for kachikoshi on the final 
   day (7-7 record). Midorifuji has already locked in a losing record."

5. PRESSURE SITUATION:
   "Kirishima is narrowly favored over Daieisho (53% confidence). Pushing 
   attacks likely at the tachiai. Kirishima is competing as ozeki. 
   Kirishima appears overranked based on recent form."
""")

## Batch Generation Function

In [ ]:
def generate_torikumi_preview(bouts_df: pd.DataFrame,
                               winner_model: lgb.Booster,
                               kimarite_model: lgb.Booster,
                               kimarite_encoder,
                               feature_cols: List[str],
                               name_lookup: Dict[int, str]) -> pd.DataFrame:
    """
    Generate narratives for a day's worth of bouts.
    
    Returns DataFrame with bout info and narrative.
    """
    X = bouts_df[feature_cols].fillna(0)
    
    winner_probs = winner_model.predict(X)
    kimarite_probs = kimarite_model.predict(X)
    
    results = []
    
    for idx, (_, row) in enumerate(bouts_df.iterrows()):
        kim_probs = {k: kimarite_probs[idx, j] 
                    for j, k in enumerate(kimarite_encoder.classes_)}
        
        narrative = generate_narrative(row, winner_probs[idx], kim_probs, name_lookup)
        
        east_name = name_lookup.get(row['eastId'], f"East")
        west_name = name_lookup.get(row['westId'], f"West")
        
        results.append({
            'bout_id': row.get('bout_id'),
            'bashoId': row.get('bashoId'),
            'day': row.get('day'),
            'east': east_name,
            'west': west_name,
            'p_east_wins': winner_probs[idx],
            'narrative': narrative
        })
    
    return pd.DataFrame(results)

In [ ]:
# Demo: Generate previews for one day
if len(recent) > 0:
    # Get bouts from one day
    latest_basho = recent['bashoId'].max()
    one_day = recent[
        (recent['bashoId'] == latest_basho) & 
        (recent['day'] == recent[recent['bashoId'] == latest_basho]['day'].max())
    ].head(15)
    
    if len(one_day) > 0:
        previews = generate_torikumi_preview(
            one_day, winner_model, kimarite_model, 
            kimarite_encoder, feature_cols, name_lookup
        )
        
        print(f"\n\n{'='*80}")
        print(f"TORIKUMI PREVIEW: {latest_basho} Day {one_day['day'].iloc[0]}")
        print(f"{'='*80}\n")
        
        for _, row in previews.iterrows():
            print(f"\n{row['east']} vs {row['west']}")
            print("-" * 40)
            print(row['narrative'])
            print()

## Summary

In [ ]:
print("""
================================================================================
NARRATIVE GENERATION COMPLETE
================================================================================

The narrative generator produces bout previews that include:

1. PREDICTION & CONFIDENCE
   - Winner prediction with probability
   - Confidence level description (heavy favorite -> coin flip)

2. BOUT TYPE
   - Derived from kimarite category probabilities
   - Push battle, belt battle, or evasive techniques

3. LIKELY TECHNIQUES
   - Top kimarite with English translations
   - Only shown if probability > 10%

4. HEAD-TO-HEAD HISTORY
   - First meeting flag
   - Historical record for frequent matchups
   - Current streak information

5. PRESSURE CONTEXT
   - Kachikoshi/makekoshi situations
   - Rank-specific pressure (ozeki, yokozuna)
   - Day 15 pressure for 7-7 records

6. RATING VS RANK
   - Flags when ELO suggests over/underranking
   - Useful for promotion/demotion narratives

To use in production:
1. Load upcoming bout data with features
2. Call generate_torikumi_preview()
3. Display or export narratives
""")

## Project Complete!

All notebooks are ready:
1. `01_data_collection.ipynb` - Fetch data from API
2. `02_rating_systems.ipynb` - Compute ELO and Glicko-2
3. `03_feature_engineering.ipynb` - Build feature set
4. `04_model_training.ipynb` - Train LightGBM models
5. `05_evaluation.ipynb` - Metrics and SHAP analysis
6. `06_narrative_generation.ipynb` - Bout preview generation